# Geant4-DNA output vertical slice

This is a compact, workflow-oriented inspection of two regenerated PureWater dumps. It makes no scientific claim: the defaults contain only two events and exist to validate the output-reading workflow.

Run the cells from top to bottom in a fresh GEANT4 kernel. Edit the two dump paths in the next cell to inspect another pair of outputs.

## 1. Configuration and input checks

The defaults point at ignored regenerated data. Required inputs fail with instructions. Mesoscopic aggregate and spatial outputs are optional and are skipped with guidance when absent. Use unprefixed dump directories containing `Manifest.json`; results-index directories and filename-prefixed dumps need resolving first.

In [1]:
from pathlib import Path
import os
import re
import sys

import pandas as pd
from IPython.display import Markdown, display

from g4utils.DnaChem import Dump, Simulation
from g4utils.HDF5 import SpeciesMesoSpatialFile

dev_dir = os.environ.get('DEV_DIR')
if not dev_dir:
    raise RuntimeError(
        'DEV_DIR is unset. Run ~/dotfiles/scripts/windows/setup-dev-env.ps1 and restart the shell before using this notebook.'
    )
DEV_DIR = Path(dev_dir)

DEFAULT_RUN_ROOT = (
    DEV_DIR
    / 'GEANT4'
    / 'SIM'
    / 'dnachem-min'
    / '.scratch'
    / 'tests'
    / '2026-11-03__read-notebooks-plan.md-and-implement-1-establish-the'
)

# Edit these two paths for a different pair of dumps.
IRT_DUMP = DEFAULT_RUN_ROOT / 'irt'
MESO_DUMP = DEFAULT_RUN_ROOT / 'meso-spatial'

def require_directory(label: str, setting: str, path: Path) -> Path:
    if not path.is_dir():
        raise FileNotFoundError(
            f'{label} dump is unavailable: {path}. Edit {setting} or regenerate the documented demo data in notebooks/README.md.'
        )
    return path

def require_file(dump_path: Path, filename: str, stage: str) -> Path:
    path = dump_path / filename
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(
            f'Required {stage} file is missing or empty: {path}. Select a dump that writes {filename}, or regenerate it with that scorer enabled.'
        )
    return path

def optional_file(dump_path: Path, filename: str, stage: str, guidance: str) -> Path | None:
    path = dump_path / filename
    if path.is_file() and path.stat().st_size > 0:
        return path
    display(Markdown(
        f'**Optional {stage} file is missing or empty:** `{path}`. This notebook will skip that optional output. {guidance}'
    ))
    return None

IRT_DUMP = require_directory('IRT', 'IRT_DUMP', IRT_DUMP)
MESO_DUMP = require_directory('mesoscopic', 'MESO_DUMP', MESO_DUMP)
for dump_path, label in ((IRT_DUMP, 'IRT'), (MESO_DUMP, 'mesoscopic')):
    require_file(dump_path, 'Manifest.json', f'{label} provenance')

irt_dump = Dump(IRT_DUMP)
meso_dump = Dump(MESO_DUMP)
physics_file = require_file(IRT_DUMP, 'PhysicsInteractions.csv', 'physical-stage')
track_file = require_file(IRT_DUMP, 'TrackLengths.csv', 'track-length')
meso_species_file = optional_file(MESO_DUMP, 'SpeciesMeso.csv', 'mesoscopic aggregate', 'Enable `/chem/meso/enable true` with IRT_syn before `/run/initialize` and regenerate to inspect aggregate counts.')
spatial_file = optional_file(MESO_DUMP, 'SpeciesMesoSpatial.h5', 'mesoscopic spatial', 'Set `/chem/meso/spatialOutput true` before `/run/initialize` and regenerate to inspect spatial metadata.')

prechemical_files = sorted(IRT_DUMP.glob('PreChemical_run*_event*.txt'))
if not prechemical_files:
    raise FileNotFoundError(
        f'No pre-chemical event files were found in {IRT_DUMP}. Regenerate with pre-chemical output enabled.'
    )

display(pd.DataFrame({
    'dump': ['IRT', 'mesoscopic'],
    'path': [str(IRT_DUMP), str(MESO_DUMP)],
    'manifest': [str(IRT_DUMP / 'Manifest.json'), str(MESO_DUMP / 'Manifest.json')],
}))

,dump,path,manifest
0,IRT,D:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2...,D:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2...
1,mesoscopic,D:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2...,D:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2...


## 2. Provenance and discovered files

`Dump` and `Simulation` provide the manifest-backed view. The inventory deliberately compares declared files with files actually present on disk.

In [2]:
def manifest_summary(label: str, dump: Dump) -> dict:
    manifest = dump.manifest
    return {
        'dump': label,
        'geant4Version': manifest.geant4Version,
        'macro': manifest.macro,
        'chemistry': manifest.chemistry,
        'timeStepModel': manifest.raw.get('timeStepModel'),
        'chemistryModel': manifest.chemistryModel,
        'handOverTime_ns': manifest.handOverTime_ns,
        'chemistryEndTime_ns': manifest.chemistryEndTime_ns,
        'mesoSpatialOutput': manifest.mesoSpatialOutput,
        'totalEvents': manifest.totalEvents,
        'totalEnergyDeposit_eV': manifest.totalEnergyDeposit_eV,
    }

def file_inventory(label: str, dump: Dump) -> pd.DataFrame:
    actual = {path.name: path for path in dump.path.iterdir() if path.is_file()}
    declared = set(dump.files)
    rows = []
    for filename in sorted(set(actual) | declared):
        path = actual.get(filename)
        rows.append({
            'dump': label,
            'file': filename,
            'declared_in_manifest': filename in declared,
            'present_on_disk': path is not None,
            'size_kib': None if path is None else round(path.stat().st_size / 1024, 1),
        })
    return pd.DataFrame(rows)

provenance = pd.DataFrame([
    manifest_summary('IRT', irt_dump),
    manifest_summary('mesoscopic', meso_dump),
])
display(provenance)
display(Markdown('### IRT manifest (including beam, seed, and settings)'))
display(irt_dump.manifest)
display(Markdown('### Mesoscopic manifest (including beam, seed, and settings)'))
display(meso_dump.manifest)

# Restrict discovery to the selected dump, avoiding unrelated siblings/indexes.
display(Simulation(IRT_DUMP).table())
display(Simulation(MESO_DUMP).table())

file_inventory_table = pd.concat([
    file_inventory('IRT', irt_dump),
    file_inventory('mesoscopic', meso_dump),
], ignore_index=True)
display(file_inventory_table)

,dump,geant4Version,macro,chemistry,timeStepModel,chemistryModel,handOverTime_ns,chemistryEndTime_ns,mesoSpatialOutput,totalEvents,totalEnergyDeposit_eV
0,IRT,geant4-11-04-patch-03,..\notebooks\demo_irt.in,PureWater,IRT_syn,IRT_syn,NaN,1000,False,2,20000
1,mesoscopic,geant4-11-04-patch-03,..\notebooks\demo_meso_spatial.in,PureWater,IRT_syn,IRT_syn+mesoscopic,5.0,1000000,True,2,20000


### IRT manifest (including beam, seed, and settings)

schemaVersion,1
timestamp,2026-11-03T17:31:29
elapsedSinceStart_s,9.7102088
elapsedSincePreviousDump_s,9.7102088
geant4Version,geant4-11-04-patch-03
macro,..\notebooks\demo_irt.in
runMode,Serial
threads,1
outputDirAsConfigured,D:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2026-11-03__read-notebooks-plan.md-and-implement-1-establish-the\irt
outputDirAbsolute,D:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2026-11-03__read-notebooks-plan.md-and-implement-1-establish-the\irt
prefix,


### Mesoscopic manifest (including beam, seed, and settings)

schemaVersion,1
timestamp,2026-11-03T17:32:30
elapsedSinceStart_s,21.0669926
elapsedSincePreviousDump_s,21.0669926
geant4Version,geant4-11-04-patch-03
macro,..\notebooks\demo_meso_spatial.in
runMode,Serial
threads,1
outputDirAsConfigured,D:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2026-11-03__read-notebooks-plan.md-and-implement-1-establish-the\meso-spatial
outputDirAbsolute,D:\DEV\GEANT4\SIM\dnachem-min\.scratch\tests\2026-11-03__read-notebooks-plan.md-and-implement-1-establish-the\meso-spatial
prefix,


,dump,chemistry,pH,totalEvents,totalEnergyDeposit_eV,particle,beamEnergy_keV,chemistryModel,handOverTime_ns,chemistryEndTime_ns,voxelSize_nm,mesoPixels,mesoTimesPerDecade,mesoSpatialOutput,threads,wallTime_s
0,irt,PureWater,7,2,20000,e-,10,IRT_syn,None,1000,None,None,None,False,1,4.855004


,dump,chemistry,pH,totalEvents,totalEnergyDeposit_eV,particle,beamEnergy_keV,chemistryModel,handOverTime_ns,chemistryEndTime_ns,voxelSize_nm,mesoPixels,mesoTimesPerDecade,mesoSpatialOutput,threads,wallTime_s
0,meso-spatial,PureWater,7,2,20000,e-,10,IRT_syn+mesoscopic,5,1000000,6.25,65536,10,True,1,19.579964


,dump,file,declared_in_manifest,present_on_disk,size_kib
0,IRT,Manifest.json,False,True,2.5
1,IRT,PhysicsInteractions.Txt,True,True,0.2
2,IRT,PhysicsInteractions.csv,True,True,0.2
3,IRT,PreChemical_run0_event0.txt,True,True,143.2
4,IRT,PreChemical_run0_event1.txt,True,True,142.0
5,IRT,Reactions.Txt,True,True,4.1
6,IRT,ReactionsMetadata.csv,True,True,0.9
7,IRT,Reactions_nt_reactions.csv,True,True,1.1
8,IRT,Species.Txt,True,True,8.8
9,IRT,Species_nt_species.csv,True,True,26.8


## 3. Visible direct readers

G4Utils already reads manifests, particle-stage ntuples, and the spatial HDF5 format. The physical-interaction, track-length, pre-chemical, and aggregate mesoscopic text/CSV formats do not have G4Utils readers, so the compact readers below remain visible in the notebook.

In [3]:
def require_columns(frame: pd.DataFrame, expected: set[str], source: Path) -> pd.DataFrame:
    missing = expected - set(frame.columns)
    if missing:
        raise ValueError(f'{source} is missing columns {sorted(missing)}; expected the documented scorer format.')
    return frame


def read_physics_interactions(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    require_columns(frame, {'label', 'count'}, path)
    frame['count'] = pd.to_numeric(frame['count'], errors='raise')
    return frame[['label', 'count']]


def read_track_lengths(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    expected = {
        'run', 'event', 'primaryLength_nm', 'primaryEkin0_keV', 'primaryEkinEnd_keV',
        'primaryEnd', 'secondaryFirstGen_nm', 'nSecondaryFirstGen',
        'secondaryAll_nm', 'nSecondaryAll',
    }
    require_columns(frame, expected, path)
    for column in expected - {'primaryEnd'}:
        frame[column] = pd.to_numeric(frame[column], errors='raise')
    return frame


PRECHEMICAL_COLUMNS = [
    'parent_id', 'molecule', 'electron_modification', 'energy_eV',
    'parent_x_nm', 'parent_y_nm', 'parent_z_nm', 'x_nm', 'y_nm', 'z_nm',
]
# G4PhysChemIO::FormattedText uses these setw widths, with no delimiter.
# Water rows omit the final position; electron rows may also omit it.
PRECHEMICAL_WIDTHS = [11, 10, 14, 13, 22, 22, 22, 14, 14, 14]


def read_prechemical_records(dump_path: Path) -> pd.DataFrame:
    frames = []
    paths = sorted(dump_path.glob('PreChemical_run*_event*.txt'))
    if not paths:
        raise FileNotFoundError(f'No pre-chemical files under {dump_path}; enable output and regenerate.')
    for path in paths:
        match = re.fullmatch(r'PreChemical_run(\d+)_event(\d+)', path.stem)
        if match is None:
            raise ValueError(f'Cannot identify run and event from {path.name}.')
        # An empty event file is valid when there were no pre-chemical records.
        if path.stat().st_size == 0:
            continue
        frame = pd.read_fwf(
            path, widths=PRECHEMICAL_WIDTHS, comment='#', header=None,
            names=PRECHEMICAL_COLUMNS,
        ).dropna(how='all')
        if frame.empty:
            continue
        for column in ['parent_id', *PRECHEMICAL_COLUMNS[3:]]:
            frame[column] = pd.to_numeric(frame[column], errors='raise')
        required = PRECHEMICAL_COLUMNS[:7]
        if frame[required].isna().any().any():
            raise ValueError(f'{path} has incomplete required pre-chemical fields.')
        frame['parent_id'] = frame['parent_id'].astype('int64')
        frame.insert(0, 'event', int(match.group(2)))
        frame.insert(0, 'run', int(match.group(1)))
        frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=['run', 'event', *PRECHEMICAL_COLUMNS])
    return pd.concat(frames, ignore_index=True)


def read_meso_species(path: Path | None) -> pd.DataFrame:
    expected = ['time_ns', 'species', 'count']
    if path is None:
        return pd.DataFrame(columns=expected)
    frame = pd.read_csv(path)
    if list(frame.columns) != expected:
        raise ValueError(f'{path} has columns {list(frame.columns)}; expected {expected}.')
    frame['time_ns'] = pd.to_numeric(frame['time_ns'], errors='raise')
    frame['count'] = pd.to_numeric(frame['count'], errors='raise')
    return frame


def spatial_metadata(path: Path | None) -> pd.DataFrame:
    columns = ['run', 'event', 'snapshots', 'first_time_ns', 'last_time_ns', 'first_cell_size_nm']
    if path is None:
        return pd.DataFrame(columns=columns)
    spatial = SpeciesMesoSpatialFile(path)
    rows = []
    for run in spatial.runs:
        for event in spatial.events(run):
            indices = spatial.snapshot_indices(run, event)
            rows.append({
                'run': run,
                'event': event,
                'snapshots': len(indices),
                'first_time_ns': spatial.snapshot_time_ns(run, event, indices[0]) if indices else None,
                'last_time_ns': spatial.snapshot_time_ns(run, event, indices[-1]) if indices else None,
                'first_cell_size_nm': spatial.snapshot_cell_size_nm(run, event, indices[0]) if indices else None,
            })
    return pd.DataFrame(rows, columns=columns)


In [4]:
physical_interactions = read_physics_interactions(physics_file)
track_lengths = read_track_lengths(track_file)
prechemical_records = read_prechemical_records(IRT_DUMP)
meso_species = read_meso_species(meso_species_file)
meso_spatial_metadata = spatial_metadata(spatial_file)

display(Markdown('### Physical interactions (aggregate over the dump)'))
display(physical_interactions)
display(Markdown('### Track lengths (one row per event)'))
display(track_lengths)
display(Markdown('### Pre-chemical records (first 10 rows; positions are nm)'))
display(prechemical_records.head(10))
display(Markdown('### Mesoscopic aggregate counts (summed over events)'))
display(meso_species.head(16))

if spatial_file is not None:
    display(Markdown('### Optional mesoscopic spatial-file metadata'))
    display(meso_spatial_metadata)

### Physical interactions (aggregate over the dump)

,label,count
0,e-_G4DNAAttachment,17
1,e-_G4DNAElastic,51251
2,e-_G4DNAElectronSolvation,999
3,e-_G4DNAExcitation,122
4,e-_G4DNAIonisation,1014
5,e-_G4DNAVibExcitation,5737


### Track lengths (one row per event)

,run,event,primaryLength_nm,primaryEkin0_keV,primaryEkinEnd_keV,primaryEnd,secondaryFirstGen_nm,nSecondaryFirstGen,secondaryAll_nm,nSecondaryAll
0,0,0,3940.635938,10.0,0.000025,stopped,4192.985256,226,6309.350231,511
1,0,1,2534.414142,10.0,0.000025,stopped,3655.026025,203,6768.150826,503


### Pre-chemical records (first 10 rows; positions are nm)

,run,event,parent_id,molecule,electron_modification,energy_eV,parent_x_nm,parent_y_nm,parent_z_nm,x_nm,y_nm,z_nm
0,0,0,1,H2O,0:4,10.79,0.000000,0.000000,2.725889,NaN,NaN,NaN
1,0,0,1,H2O,0:4,10.79,-0.500397,-1.097857,98.859680,NaN,NaN,NaN
2,0,0,1,H2O,0:4,10.79,-0.520728,-0.948461,110.041300,NaN,NaN,NaN
3,0,0,1,H2O,0:1,32.30,2.060651,0.622172,126.331100,NaN,NaN,NaN
4,0,0,1,H2O,0:1,32.30,30.908060,13.303170,210.642400,NaN,NaN,NaN
5,0,0,1,H2O,0:4,10.79,41.423710,18.044470,232.077200,NaN,NaN,NaN
6,0,0,1,H2O,0:3,13.39,51.384810,21.855190,253.103700,NaN,NaN,NaN
7,0,0,1,H2O,0:3,13.39,62.214840,24.824230,273.455800,NaN,NaN,NaN
8,0,0,1,H2O,0:3,13.39,65.369880,25.632880,278.363500,NaN,NaN,NaN
9,0,0,1,H2O,0:3,13.39,66.592230,25.962400,280.255200,NaN,NaN,NaN


### Mesoscopic aggregate counts (summed over events)

,time_ns,species,count
0,5.000000,H2O2^0,97
1,5.000000,H3O^1,643
2,5.000000,HO_2^-1,3
3,5.000000,HO_2°^0,13
4,5.000000,H^0,153
5,5.000000,H_2^0,100
6,5.000000,OH^-1,83
7,5.000000,O^-1,11
8,5.000000,O_2^0,4
9,5.000000,e_aq^-1,548


### Optional mesoscopic spatial-file metadata

,run,event,snapshots,first_time_ns,last_time_ns,first_cell_size_nm
0,0,0,55,5.0,1000000.0,15.258789
1,0,1,55,5.0,1000000.0,15.258789


In [5]:
assert len(track_lengths) == irt_dump.manifest.totalEvents
assert not physical_interactions.empty
assert not prechemical_records.empty
assert prechemical_records[['energy_eV', 'parent_x_nm', 'parent_y_nm', 'parent_z_nm']].notna().all().all()
electrons_with_positions = prechemical_records.query("molecule == 'e_aq'").dropna(subset=['x_nm', 'y_nm', 'z_nm'])
print(f'Discovery checks: PASS; {len(track_lengths)} events, {len(prechemical_records)} pre-chemical rows, '
      f'{len(electrons_with_positions)} solvated-electron positions.')
print(f'Kernel: {sys.executable}')


Discovery checks: PASS; 2 events, 2152 pre-chemical rows, 999 solvated-electron positions.
Kernel: E:\miniconda3\envs\GEANT4\python.exe
